In [1]:
import os
import random
from utils.hand_model_lite import HandModelMJCFLite 
from utils.hand_model import HandModel_Mujoco, HandModel
import numpy as np
import transforms3d
import torch
import trimesh
import json
import plotly.graph_objects as go
from utils.initializations import initialize_table_top

/opt/conda/envs/dexgraspnet/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here you need to choose your `hand_name`

In [2]:
mesh_path = "../data/meshdata"
hand_name ="shadow_dexee"
# hand_name ="DIP-Flex_opened_kinematics_simpl"
# hand_name ="DIP-Flex_opened_kinematics"
# hand_name = "robotiq_2"
# hand_name = "panda"

use_visual_mesh = True

if hand_name =="shadow_dexee":
    '''For shadow dexee'''
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]

elif hand_name =="barret":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]

elif hand_name == "robotiq_2":
    '''For robotiq'''
    data_path = "../data/dataset/robotiq_2/"
    hand_file = "mjcf/robotiq_2 simpl.xml"
    joint_names = [
                    "left_spring_link_joint", "left_follower",
                    "right_spring_link_joint", "right_follower_joint"
    ]

elif hand_name == "panda":
    '''For panda'''
    data_path = "../data/dataset/panda/"
    hand_file = "mjcf/panda.xml"
    joint_names = [
                    "finger_joint1", "finger_joint2"
    ]

elif hand_name == "shadow_dex_ee_simpl":
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee simpl.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]
elif hand_name =="barret_simpl":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret_simpl.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics_simpl":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics simpl.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]
 
    

translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']


In [3]:
hand_config = json.load(open('mjcf/' + hand_name + '/' + hand_name + '.json', 'r'))
device = "cpu"

In [4]:


hand_model = HandModel(
    hand_config=hand_config,
    mjcf_path='mjcf/' + hand_name + "/" + hand_name + '_simpl.xml',
    mesh_path='mjcf/assets/' + hand_name,
    contact_points_path='mjcf/' + hand_name + "/" + "contact_points_" + hand_name +  '.json',
    penetration_points_path='mjcf/' + hand_name  + "/" + "penetration_points_" + hand_name +  '.json',
    n_surface_points=200,
    device=device
)

/workspace/grasp_generation/utils/hand_model.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(visual.geom_param[1], dtype=torch.float, device=device)


In [ ]:
from utils.initializations import initialize_convex_hull
from utils.object_model import ObjectModel
from types import SimpleNamespace

data_path = "../data/BIG_BOWLS_CLASS"

object_scales_path = os.path.join(data_path, 'object_scales.json')
with open(object_scales_path, 'r') as f:
    object_scale_dict = json.load(f)

object_model = ObjectModel(
    data_root_path=data_path,
    batch_size_each=50,
    num_samples=2000, 
    device=device,
    scale_dict=object_scale_dict
)


object_code_list_all = [f for f in os.listdir(data_path) 
                        if not os.path.isfile(os.path.join(data_path, f))]
print(object_code_list_all)
object_model.initialize( [object_code_list_all[3]] )

args = SimpleNamespace(
    num_points=10,
    use_gpu=False,
    distance_lower=0.3,
    distance_upper=0.4,
    theta_lower=-np.pi / 6,
    theta_upper= np.pi / 6,
    jitter_strength=0.1,
    n_contact=100,
)

initialize_convex_hull(hand_config['init_pos'], hand_model, object_model, args)


FileNotFoundError: [Errno 2] No such file or directory: '../data/BIG_BOWLS/object_scales.json'

In [ ]:
from utils.rot6d import compute_rotation_matrix_from_ortho6d

pose_index = 4
n_hands = hand_model.hand_pose.shape[0]
translate_wirst = hand_model.hand_pose[pose_index, :3]
rotation_wirst = hand_model.hand_pose[pose_index, 3:9]


all_hand_traces = hand_model.get_plotly_data(i=pose_index, opacity=0.8, color="lightblue")

def generate_wrist_visualization(translate_wirst, rotation_wirst):
    rot_mat = compute_rotation_matrix_from_ortho6d(rotation_wirst.unsqueeze(0))[0]
    axis_vec = rot_mat[:, 2]
    vec_length = 0.1
    end_point = translate_wirst + vec_length * axis_vec

    wrist_trace = go.Scatter3d(
    x=[translate_wirst[0].item()],
    y=[translate_wirst[1].item()],
    z=[translate_wirst[2].item()],
    mode="markers",
    marker=dict(size=6, color="red"),
    name="translate_wirst",
)


    vector_trace = go.Scatter3d(
    x=[translate_wirst[0].item(), end_point[0].item()],
    y=[translate_wirst[1].item(), end_point[1].item()],
    z=[translate_wirst[2].item(), end_point[2].item()],
    mode="lines",
    line=dict(color="green", width=6),
    name="wrist_x_axis",
)
    
    return wrist_trace,vector_trace


contacts_1 = hand_model.get_contact_candidates()
contacts_1[0]

contacts_sas = []
for contact_i in contacts_1[0]:
    one_point = go.Scatter3d(
        x=[contact_i[0].detach().numpy()],
        y=[contact_i[1].detach().numpy()],
        z=[contact_i[2].detach().numpy()],
        mode="markers",
        marker=dict(size=3, color="blue"),
        name="sss")
    contacts_sas.append(one_point)





vectors_data = []
for i in range(n_hands):
    translate_wirst = hand_model.hand_pose[i, :3]
    rotation_wirst = hand_model.hand_pose[i, 3:9]
    wrist_trace, vector_trace = generate_wrist_visualization(translate_wirst, rotation_wirst)

    vectors_data.extend([wrist_trace, vector_trace])
 

fig = go.Figure(data= hand_model.get_plotly_data(0) + object_model.get_plotly_data(i=2, opacity=1)  + vectors_data)
fig.update_layout(scene_aspectmode="data")
fig.show()

object_model.object_mesh_list

[<trimesh.Trimesh(vertices.shape=(2093, 3), faces.shape=(4134, 3), name=`decomposed.obj`)>]